# scRNA-seq Preprocessing Pipeline
**Dataset**: GSE127465 — Lung cancer tumor-infiltrating immune cells  
**Goal**: Raw count → QC → Normalization → HVG → PCA → Harmony → UMAP → Cell type annotation

---
## Pipeline Overview
```
RAW counts (26 samples)
    └─ 1. Load & merge samples
    └─ 2. QC  (hard cutoff → MAD filter)
    └─ 3. Normalization  (CPM + log1p)
    └─ 4. HVG selection  (top 2,000 genes)
    └─ 5. PCA  (arpack, 50 PCs)
    └─ 6. Batch correction  (Harmony)
    └─ 7. UMAP + Leiden clustering
    └─ 8. Cell type annotation
```

## 0. Environment Setup
```bash
conda create -n spatial python=3.10
conda activate spatial
pip install scanpy harmonypy scikit-misc tqdm
```

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import harmonypy as hm
import glob
from tqdm import tqdm
from scipy.stats import median_abs_deviation

sc.settings.verbosity = 1
print(f"scanpy {sc.__version__}")

## 1. Load & Merge Raw Samples

26개의 human sample TSV 파일을 AnnData로 병합한다.

**메모리 최적화 포인트**
- `float32` 캐스팅으로 float64 대비 메모리 50% 절감
- CSR sparse matrix: scRNA-seq 특성상 대부분의 값이 0이므로 0이 아닌 좌표+값만 저장
- 루프마다 `del df`로 즉시 해제
- `index_unique='-'`로 샘플 간 barcode 중복 방지 (`bcIHWD` → `bcIHWD-0`)

In [ ]:
DATA_DIR = 'dataset/raw/GSE127465_RAW'
H5AD_PATH = 'dataset/raw/GSE127465_human_all.h5ad'

files = glob.glob(f'{DATA_DIR}/*human*.tsv.gz')
print(f"Found {len(files)} sample files")

# 데이터 구조 확인 (최초 1회)
df_check = pd.read_csv(files[0], sep='\t', index_col=0, compression='gzip')
print(f"Shape: {df_check.shape}  |  index=cells, columns=genes")
del df_check

# 전체 샘플 병합
adata_list = []
for f in tqdm(files, desc="Loading samples"):
    df = pd.read_csv(f, sep='\t', index_col=0, compression='gzip')
    adata = sc.AnnData(sp.csr_matrix(df.values.astype('float32')))
    adata.obs_names = df.index.tolist()
    adata.var_names = df.columns.tolist()
    adata_list.append(adata)
    del df

adata_raw = sc.concat(adata_list, join='outer', index_unique='-')
del adata_list
print(f"Merged: {adata_raw.shape[0]:,} cells × {adata_raw.shape[1]:,} genes")

adata_raw.write_h5ad(H5AD_PATH)
print(f"Saved → {H5AD_PATH}")

## 2. Quality Control

### QC 핵심 지표 3가지
| 지표 | 의미 | 이상치 기준 |
|------|------|------------|
| `total_counts` | 세포당 총 UMI count | 너무 낮으면 사멸 세포, 너무 높으면 doublet |
| `n_genes_by_counts` | 세포당 발현 유전자 수 | 너무 낮으면 empty droplet |
| `pct_counts_mt` | 미토콘드리아 비율 | 높으면 세포막 파괴 → 세포질 mRNA 유출 |

### 필터링 전략: Hard cutoff → MAD
MAD 단독 적용 시 low-count 세포가 다수를 차지하면 median 자체가 낮게 형성되어  
low-count 세포도 정상 판정되는 문제 발생 → **hard cutoff를 선적용하여 median을 끌어올린 뒤 MAD 적용**

In [ ]:
adata = sc.read_h5ad('dataset/raw/GSE127465_human_all.h5ad')
print(adata)

# QC 지표 계산
adata.var['mt']   = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))
adata.var['hb']   = adata.var_names.str.contains(r'^HB[ABDEGMQZ]\d*(?!\w)')

sc.pp.calculate_qc_metrics(
    adata, qc_vars=['mt', 'ribo', 'hb'],
    percent_top=None, log1p=True, inplace=True
)

# QC 전 분포 확인
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, show=True)

In [ ]:
# Step 1: Hard cutoff (MAD 정상 작동을 위한 선처리)
n_before = adata.n_obs
adata = adata[adata.obs['n_genes_by_counts'] >= 200].copy()
adata = adata[adata.obs['total_counts']      >= 500].copy()
print(f"Hard cutoff: {n_before:,} → {adata.n_obs:,} cells")

# Step 2: MAD-based outlier filter
def is_outlier(adata, metric: str, nmads: int) -> pd.Series:
    """중앙값 ± (nmads × MAD) 범위를 벗어난 세포를 True로 반환"""
    M = adata.obs[metric]
    return (M < np.median(M) - nmads * median_abs_deviation(M)) |            (np.median(M) + nmads * median_abs_deviation(M) < M)

adata.obs['outlier']    = (is_outlier(adata, 'log1p_total_counts', 5) |
                           is_outlier(adata, 'log1p_n_genes_by_counts', 5))
adata.obs['mt_outlier'] = (is_outlier(adata, 'pct_counts_mt', 3) |
                           (adata.obs['pct_counts_mt'] > 20))

print(f"MAD outlier:  {adata.obs['outlier'].sum():,} cells flagged")
print(f"MT outlier:   {adata.obs['mt_outlier'].sum():,} cells flagged")

n_before = adata.n_obs
adata = adata[(~adata.obs['outlier']) & (~adata.obs['mt_outlier'])].copy()
print(f"After MAD filter: {n_before:,} → {adata.n_obs:,} cells")

# QC 후 분포 확인
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, show=True)

## 3. Normalization

세포마다 sequencing depth가 다르므로 모든 세포를 동일한 기준(10,000 CPM)으로 맞춘 뒤  
`log1p` 변환으로 극단값을 압축한다. (`log1p`: count=0인 세포의 log(0) → -∞ 방지)

raw count는 `adata.layers['counts']`에 보존 → 이후 DEG 분석에서 활용

In [ ]:
# 0-count 세포 제거 (normalize_total의 0 나누기 방지용 방어 로직)
sc.pp.filter_cells(adata, min_counts=1)

# raw count 보존
adata.layers['counts'] = adata.X.copy()

# CPM normalization + log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(adata)

## 4. Highly Variable Gene (HVG) Selection

41,861개 유전자 중 **세포마다 발현 차이가 큰 상위 2,000개**만 선택한다.  
모든 세포에서 균일하게 발현되는 유전자는 세포 타입 구분에 기여하지 않으므로 제거한다.

`n_top_genes=2000`: Seurat/Scanpy 공식 기본값. 너무 적으면 정보 손실, 너무 많으면 노이즈 증가.  
`flavor='seurat_v3'`: raw count 기반 분산 추정 → `layer='counts'` 지정 필요

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000,
                            flavor='seurat_v3', layer='counts')
print(f"HVG: {adata.var.highly_variable.sum():,} / {adata.n_vars:,} genes selected")
sc.pl.highly_variable_genes(adata)

# annotation 시 HVG 외 marker gene 조회를 위해 전체 유전자 백업
adata.raw = adata

# HVG만 남기기
adata = adata[:, adata.var.highly_variable].copy()
print(f"After HVG filter: {adata.shape}")

## 5. PCA

2,000개 유전자 공간을 50개의 주성분(PC)으로 압축한다.  
`pca_variance_ratio` 플롯에서 elbow 지점(분산 기여도가 급감하는 지점)까지의 PC 수를  
Harmony → UMAP의 `n_pcs`로 사용한다.

`svd_solver='arpack'`: 대용량 sparse matrix에 최적화된 방식 (iterative partial SVD)

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)
# → elbow 확인 후 아래 n_pcs 결정 (본 데이터: PC17 이후 완만 → n_pcs=17)

## 6. Batch Correction (Harmony)

26개 샘플은 서로 다른 환자·시점에서 수집되어 **batch effect** 가 존재한다.  
batch effect 방치 시 생물학적 유사성이 아닌 샘플 출처로 클러스터가 형성된다.

Harmony는 PCA 임베딩을 샘플별로 반복 보정하여 sample-agnostic한 표현을 생성한다.  
보정 결과는 `adata.obsm['X_pca_harmony']`에 저장된다.

In [ ]:
# 샘플 정보 추출 (barcode 접미사에 샘플 번호가 인코딩됨: 'bcXXXX-2' → sample='2')
adata.obs['sample'] = adata.obs_names.str.split('-').str[-1]
print(adata.obs['sample'].value_counts())

# Harmony batch correction
pca_result = adata.obsm['X_pca']
meta       = adata.obs[['sample']]
ho         = hm.run_harmony(pca_result, meta, 'sample')
adata.obsm['X_pca_harmony'] = ho.Z_corr
print("Harmony done.")

## 7. UMAP + Leiden Clustering

**UMAP**: Harmony로 보정된 임베딩을 2D로 펼쳐 시각화한다. 비슷한 유전자 발현 패턴의 세포가 근접 배치된다.  
**Leiden**: graph 기반 클러스터링 알고리즘. `resolution`으로 클러스터 세분화 정도 조절.  
`random_state=42`로 UMAP 재현성 고정.

`n_pcs=17`: PCA elbow plot 기준

In [ ]:
N_PCS = 17  # PCA variance ratio elbow 기준

sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_pcs=N_PCS)
sc.tl.umap(adata, random_state=42)
sc.tl.leiden(adata, resolution=0.5)

print(f"Clusters: {adata.obs['leiden'].nunique()}")
sc.pl.umap(adata, color=['leiden', 'sample'], ncols=2)

## 8. Cell Type Annotation

알려진 marker gene 발현량을 UMAP에 overlay하여 각 클러스터의 세포 타입을 확인한다.  
`adata.raw`에 백업된 전체 유전자에서 marker를 조회하므로 HVG에 없는 유전자도 확인 가능.

| Marker | Cell type |
|--------|-----------|
| CD3D   | T cell    |
| CD68   | Macrophage |
| EPCAM  | Cancer cell (epithelial) |
| CD19   | B cell    |

In [ ]:
# Marker gene 발현량 확인
sc.pl.umap(adata, color=['CD3D', 'CD68', 'EPCAM', 'CD19'], ncols=2, use_raw=True)
sc.pl.umap(adata, color='leiden')

In [ ]:
# 클러스터 → 세포 타입 매핑
# Unknown: 단일 marker로 판단이 불명확한 클러스터 (Phase 2에서 세분화 예정)
cell_type_map = {
    '0': 'T cell',
    '5': 'T cell',
    '2': 'Macrophage',
    '4': 'Macrophage',
    '3': 'B cell',
    '7': 'Cancer cell',
    '9': 'Cancer cell',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map).fillna('Unknown')

print(adata.obs['cell_type'].value_counts())
sc.pl.umap(adata, color='cell_type', title='Cell Type Annotation (Phase 1)')

In [ ]:
adata.write_h5ad('dataset/GSE127465_human_final.h5ad')
print("Saved → dataset/GSE127465_human_final.h5ad")
print(adata)